<a href="https://colab.research.google.com/github/CemKeremSahin/NIR-to-RGB-Colorization/blob/main/Notebooks/MUGAN_Train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
print(f"Sahip olduğun CPU çekirdek sayısı: {os.cpu_count()}")

In [ ]:
# --- 1. COLAB KURULUMLARI  DRIVE BAĞLANTISI ---
import os

# Gerekli kütüphaneyi kuruyoruz
try:
    import segmentation_models_pytorch as smp
    print("Kütüphane zaten kurulu.")
except ImportError:
    print("Kütüphane kuruluyor...")
    os.system('pip install segmentation_models_pytorch')
    import segmentation_models_pytorch as smp

In [ ]:
# DRIVE BAĞLANTISI
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Drive dan colab e aktarma
!cp -r /content/drive/MyDrive/realword_dataset/train /content/

In [ ]:
import os
# Ayarladığınız yolu buraya yazın
NIR_FOLDER = "/content/train/NIR"
RGB_FOLDER = "/content/train/RGB"

print("--- VERİ SETİ KONTROLÜ BAŞLIYOR ---\n")

def veri_kontrol():
    # NIR Klasörünü kontrol et
    if not os.path.exists(NIR_FOLDER):
        print(f"HATA: {NIR_FOLDER} klasörü bulunamadı! Lütfen '!cp' komutunu kontrol et.")
        return

    # Sadece .bmp dosyalarını listele ve sırala
    nir_list = sorted([f for f in os.listdir(NIR_FOLDER) if f.lower().endswith('.bmp')])
    print(f"NIR klasöründe {len(nir_list)} adet .bmp dosyası bulundu.")

    # RGB Klasörünü kontrol et (Eğer varsa)
    if os.path.exists(RGB_FOLDER):
        rgb_list = sorted([f for f in os.listdir(RGB_FOLDER) if f.lower().endswith('.bmp')])
        print(f"RGB klasöründe {len(rgb_list)} adet .bmp dosyası bulundu.")

        # Makale verisiyle karşılaştırma
        if len(nir_list) == 538:
            print(">>> DURUM: Makaledeki 'Real-World' veri seti sayısına (538) tam uyuyor! [cite: 387]")

        # Eşleşme kontrolü
        if len(nir_list) == len(rgb_list):
            print(">>> Başarılı: NIR ve RGB dosya sayıları eşit.")
        else:
            print(f">>> UYARI: Dosya sayıları farklı! (NIR: {len(nir_list)}, RGB: {len(rgb_list)})")

        print("\nÖrnek Eşleşmeler (İlk 5):")
        for i in range(min(5, len(nir_list))):
            print(f"  {nir_list[i]} <---> {rgb_list[i] if i < len(rgb_list) else 'Eksik!'}")
    else:
        print(f"NOT: {RGB_FOLDER} klasörü bulunamadı. Sadece NIR dosyaları işlenebilir.")

veri_kontrol()

--- VERİ SETİ KONTROLÜ BAŞLIYOR ---

NIR klasöründe 458 adet .bmp dosyası bulundu.
RGB klasöründe 458 adet .bmp dosyası bulundu.
>>> Başarılı: NIR ve RGB dosya sayıları eşit.

Örnek Eşleşmeler (İlk 5):
  0001.bmp <---> 0001.bmp
  0002.bmp <---> 0002.bmp
  0003.bmp <---> 0003.bmp
  0004.bmp <---> 0004.bmp
  0005.bmp <---> 0005.bmp


In [ ]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from tqdm import tqdm

# --- 1. PERFORMANS AYARLARI ---
torch.backends.cuda.matmul.fp32_precision = 'high'
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. AYARLAR VE YOLLAR ---
ROOT_PATH = "/content/train"
SAVE_PATH = "/content/drive/MyDrive/S-NET/Weights"
os.makedirs(SAVE_PATH, exist_ok=True)

PATCH_SIZE = 256
BATCH_SIZE = 16
NUM_EPOCHS = 3500
SAVE_EVERY_N_EPOCHS = 500

# --- 3. MUGAN MIMARISI (GENERATOR) ---
class MixedSkipBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(MixedSkipBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class MUGANGenerator(nn.Module):
    def __init__(self, in_channels=1, out_channels=3):
        super(MUGANGenerator, self).__init__()
        self.enc1 = MixedSkipBlock(in_channels, 64)
        self.enc2 = MixedSkipBlock(64, 128)
        self.enc3 = MixedSkipBlock(128, 256)
        self.enc4 = MixedSkipBlock(256, 512)
        self.pool = nn.MaxPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.bridge = MixedSkipBlock(512, 1024)
        self.dec4 = MixedSkipBlock(1024 + 512, 512)
        self.dec3 = MixedSkipBlock(512 + 256, 256)
        self.dec2 = MixedSkipBlock(256 + 128, 128)
        self.dec1 = MixedSkipBlock(128 + 64, 64)
        self.final = nn.Sequential(nn.Conv2d(64, out_channels, kernel_size=1), nn.Tanh())

    def forward(self, x):
        s1 = self.enc1(x); p1 = self.pool(s1)
        s2 = self.enc2(p1); p2 = self.pool(s2)
        s3 = self.enc3(p2); p3 = self.pool(s3)
        s4 = self.enc4(p3); p4 = self.pool(s4)
        b = self.bridge(p4)
        d4 = self.dec4(torch.cat([self.upsample(b), s4], dim=1))
        d3 = self.dec3(torch.cat([self.upsample(d4), s3], dim=1))
        d2 = self.dec2(torch.cat([self.upsample(d3), s2], dim=1))
        d1 = self.dec1(torch.cat([self.upsample(d2), s1], dim=1))
        return self.final(d1)

# --- 4. DISCRIMINATOR (PATCHGAN) ---
class PatchGAN(nn.Module):
    def __init__(self):
        super().__init__()
        def block(in_f, out_f, stride=2, norm=True):
            layers = [nn.Conv2d(in_f, out_f, 4, stride, 1)]
            if norm: layers.append(nn.InstanceNorm2d(out_f))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return nn.Sequential(*layers)
        self.model = nn.Sequential(
            block(4, 64, norm=False),
            block(64, 128), block(128, 256), block(256, 512, stride=1),
            nn.Conv2d(512, 1, 4, 1, 1)
        )
    def forward(self, ir, rgb): return self.model(torch.cat((ir, rgb), 1))

# --- 5. DATASET VE LOSS ---
class SingleBandNIRDatasetRAM(Dataset):
    def __init__(self, root_path, patch_size=256):
        self.patch_size = patch_size
        self.data_cache = []
        nir_dir = os.path.join(root_path, "NIR")
        rgb_dir = os.path.join(root_path, "RGB")
        nir_files = sorted([f for f in os.listdir(nir_dir) if f.lower().endswith('.bmp')])
        rgb_files = sorted([f for f in os.listdir(rgb_dir) if f.lower().endswith('.bmp')])

        print(f"📥 {len(nir_files)} adet veri RAM'e yükleniyor...")
        for n_file, r_file in tqdm(zip(nir_files, rgb_files), total=len(nir_files)):
            ir_img = cv2.imread(os.path.join(nir_dir, n_file), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
            rgb_img = cv2.imread(os.path.join(rgb_dir, r_file), cv2.IMREAD_COLOR)
            rgb_img = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            self.data_cache.append({'ir': np.expand_dims(ir_img, axis=-1), 'rgb': rgb_img})

    def __len__(self): return len(self.data_cache)
    def __getitem__(self, idx):
        d = self.data_cache[idx]
        ir = torch.from_numpy(d['ir']).permute(2, 0, 1) * 2.0 - 1.0
        rgb = torch.from_numpy(d['rgb']).permute(2, 0, 1) * 2.0 - 1.0
        i, j, h, w = transforms.RandomCrop.get_params(ir, (self.patch_size, self.patch_size))
        return transforms.functional.crop(ir, i, j, h, w), transforms.functional.crop(rgb, i, j, h, w)

class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.vgg = models.vgg16(weights='DEFAULT').features[:16].eval().to(DEVICE)
        for p in self.vgg.parameters(): p.requires_grad = False
        self.criterion = nn.L1Loss()
    def forward(self, fake, real): return self.criterion(self.vgg((fake+1)/2), self.vgg((real+1)/2))

# --- 6. EĞİTİM DÖNGÜSÜ ---
model_G = MUGANGenerator(1, 3).to(DEVICE)
model_D = PatchGAN().to(DEVICE)

optimizer_G = optim.Adam(model_G.parameters(), lr=1e-4, betas=(0.5, 0.999))
optimizer_D = optim.Adam(model_D.parameters(), lr=1e-4, betas=(0.5, 0.999))

criterion_GAN = nn.BCEWithLogitsLoss()
criterion_MAE = nn.L1Loss()
criterion_percep = PerceptualLoss()

loader = DataLoader(SingleBandNIRDatasetRAM(ROOT_PATH, PATCH_SIZE), BATCH_SIZE, shuffle=True, pin_memory=True)

print(f"\n🚀 MUGAN Eğitimi Başlıyor...")

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_g, epoch_d = 0, 0
    loop = tqdm(loader, leave=False)
    for ir, real in loop:
        ir, real = ir.to(DEVICE), real.to(DEVICE)

        # DISCRIMINATOR
        optimizer_D.zero_grad()
        fake = model_G(ir).detach()
        l_d = (criterion_GAN(model_D(ir, real), torch.ones_like(model_D(ir, real))) +
               criterion_GAN(model_D(ir, fake), torch.zeros_like(model_D(ir, fake)))) * 0.5
        l_d.backward(); optimizer_D.step()

        # GENERATOR
        optimizer_G.zero_grad()
        fake = model_G(ir)
        l_g_mae = criterion_MAE(fake, real) * 50.0
        l_g_percep = criterion_percep(fake, real) * 10.0
        l_g_gan = criterion_GAN(model_D(ir, fake), torch.ones_like(model_D(ir, fake)))

        l_g = l_g_gan + l_g_mae + l_g_percep
        l_g.backward(); optimizer_G.step()

        epoch_g += l_g.item(); epoch_d += l_d.item()
        loop.set_description(f"Epoch [{epoch}/{NUM_EPOCHS}]")
        loop.set_postfix(G=l_g.item(), D=l_d.item())

    if epoch % 100 == 0: print(f"Epoch {epoch} | G_Loss: {epoch_g/len(loader):.4f}")
    if epoch % SAVE_EVERY_N_EPOCHS == 0:
      torch.save({
        'epoch': epoch,
        'model_G_state_dict': model_G.state_dict(),
        'model_D_state_dict': model_D.state_dict(),
        'optimizer_G_state_dict': optimizer_G.state_dict(),
        'optimizer_D_state_dict': optimizer_D.state_dict(),
      }, os.path.join(SAVE_PATH, f"mugan_epoch_{epoch}.pth"))

print("✅ Eğitim bitti. Oturum otomatik olarak kapatılıyor...")
runtime.unassign()